# 09b — PRIO-GRID Feature Engineering Methods

**Depends on:** notebook 09 (PRIO-GRID pull), notebook 04 (UCDP-GED events)  
**Reads from ADLS:** `raw/prio_grid/`, `raw/ucdp_ged/`  
**Writes to ADLS:** `raw/prio_grid/{RUN_DATE}/priogrid_engineered_features.parquet`  

## Purpose

Notebook 09 produces a simple country-year aggregation (mean/sum/max per variable).
This notebook builds seven additional feature families from the same grid-cell data
that are more predictive of instability outcomes because they capture *within-country
variation* — the distributional shape, spatial concentration, and temporal shocks
that flat country-level averages erase.

All outputs are at **country-year** resolution and are ready to join into the
feature matrix built in notebook 14.

## Seven methods

| # | Method | Instability mechanism |
|---|---|---|
| 1 | **Distributional aggregates** | Dispersion of NTL/pop captures urban–rural fractures |
| 2 | **NTL Gini / darkness fraction** | Within-country economic inequality proxy |
| 3 | **Population-weighted climate stress** | Which populations face rainfall/temp volatility |
| 4 | **Conflict cell density (UCDP-GED join)** | Spatial breadth of active violence |
| 5 | **Lootable resource population exposure** | Grievance potential of resource enclaves |
| 6 | **Economic polarization (core vs. periphery)** | Primate city concentration, regional neglect |
| 7 | **NTL economic shock index** | Sudden economic deterioration signal |

## Required environment variables
```
ADLS_ACCOUNT_NAME
ADLS_CONTAINER  (default: 'data')
```

In [ ]:
import os
import re
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from azure.identity import DefaultAzureCredential
import adlfs

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 40)

## Configuration

In [ ]:
ADLS_ACCOUNT_NAME = os.environ["ADLS_ACCOUNT_NAME"]
ADLS_CONTAINER    = os.getenv("ADLS_CONTAINER", "data")
RUN_DATE          = datetime.utcnow().strftime("%Y%m%d")

PANEL_START_YEAR = 2000
PANEL_END_YEAR   = 2024

# NTL thresholds (DMSP-OLS calibrated scale: 0–63)
NTL_DARK_THRESHOLD   = 3    # below this = essentially unlit
NTL_URBAN_THRESHOLD  = 20   # above this = urban / semi-urban

# Climate stress: top-quartile rainfall SD within country-year is "high stress"
CLIMATE_STRESS_QUANTILE = 0.75

# NTL shock: flag if pop-weighted NTL drops more than this many SDs below 5-yr baseline
NTL_SHOCK_THRESHOLD = 1.5

print(f"Run date     : {RUN_DATE}")
print(f"Panel years  : {PANEL_START_YEAR}–{PANEL_END_YEAR}")

## ADLS helpers

In [ ]:
credential = DefaultAzureCredential()
storage_options = {
    "account_name": ADLS_ACCOUNT_NAME,
    "credential":   credential,
}

def adls_path(subpath: str) -> str:
    return (
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}"
        f".dfs.core.windows.net/{subpath}"
    )

def write_parquet(df: pd.DataFrame, subpath: str) -> None:
    path = adls_path(subpath)
    df.to_parquet(path, storage_options=storage_options, index=False, engine="pyarrow")
    print(f"  Written {len(df):,} rows → {path}")

def read_latest_parquet(prefix: str, filename_hint: str = "") -> pd.DataFrame | None:
    """Read the parquet from the lexicographically latest date partition under prefix."""
    fs = adlfs.AzureBlobFileSystem(
        account_name=ADLS_ACCOUNT_NAME, credential=credential
    )
    full_prefix = f"{ADLS_CONTAINER}/{prefix}"
    try:
        entries = fs.ls(full_prefix, detail=False)
    except FileNotFoundError:
        print(f"  WARNING: {full_prefix} not found")
        return None

    date_dirs = sorted(
        [e for e in entries if re.search(r'/\d{8}(/|$)', e)],
        reverse=True,
    )
    if not date_dirs:
        print(f"  WARNING: no date partitions under {full_prefix}")
        return None

    pattern = f"{date_dirs[0]}/*{filename_hint}*.parquet" if filename_hint else f"{date_dirs[0]}/*.parquet"
    files = [
        f"abfss://{ADLS_CONTAINER}@{ADLS_ACCOUNT_NAME}.dfs.core.windows.net/"
        + f.replace(f"{ADLS_CONTAINER}/", "", 1)
        for f in fs.glob(pattern)
    ]
    if not files:
        print(f"  WARNING: no parquet files at {date_dirs[0]}")
        return None

    dfs = [pd.read_parquet(p, storage_options=storage_options) for p in files]
    df  = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]
    print(f"  Loaded {len(df):,} rows ← {date_dirs[0]}")
    return df

## Load PRIO-GRID data

We need two tables produced by notebook 09:
- **Static** — one row per grid cell with geographic invariants (`bdist1`, `mountains_mean`, etc.)
- **Yearly** — one row per (cell, year) with time-varying variables (`pop_gpw_sum`, `nlights_calib_mean`, etc.)

And one table from notebook 04:
- **UCDP-GED events** — one row per conflict event with a `priogrid_gid` field, used in Method 4.

In [ ]:
# ── PRIO-GRID static ──────────────────────────────────────────────────────────
df_static = read_latest_parquet("raw/prio_grid", filename_hint="static")

# ── PRIO-GRID yearly ─────────────────────────────────────────────────────────
df_yearly = read_latest_parquet("raw/prio_grid", filename_hint="yearly")

# ── UCDP-GED events (for Method 4) ───────────────────────────────────────────
df_ged = read_latest_parquet("raw/ucdp_ged", filename_hint="events")

for name, df in [("static", df_static), ("yearly", df_yearly), ("ged", df_ged)]:
    if df is not None:
        df.columns = [c.lower().strip() for c in df.columns]
        print(f"  {name:8s}: {df.shape}  cols={list(df.columns[:8])}...")
    else:
        print(f"  {name:8s}: NOT LOADED — some methods will be skipped")

## Country crosswalk (GW code → ISO3)

PRIO-GRID uses Gleditsch-Ward (`gwno`) country codes. We map these to ISO3 for joining
to the country-year feature matrix.

In [ ]:
_cw_path = next(
    (p for p in [Path("../data/country_crosswalk.csv"), Path("data/country_crosswalk.csv")]
     if p.exists()),
    None,
)
if _cw_path:
    df_cw = pd.read_csv(_cw_path, dtype=str)
    df_cw["gw_numeric"] = pd.to_numeric(df_cw["gw_numeric"], errors="coerce")
    gw_to_iso3 = dict(zip(df_cw["gw_numeric"], df_cw["iso3"]))
    print(f"Crosswalk loaded: {len(df_cw)} countries")
else:
    print("WARNING: country_crosswalk.csv not found — gwno→iso3 mapping unavailable")
    gw_to_iso3 = {}

## Merge static variables into yearly panel

Joins geographic invariants (border distance, terrain, forest cover) onto the yearly
time-varying rows so that every cell-year has both geographic and temporal attributes.
Also maps `gwno` → `iso3` and filters to the panel years.

In [ ]:
df_panel = None

if df_yearly is not None:
    df_panel = df_yearly.copy()

    # Merge static geography onto yearly rows
    if df_static is not None:
        static_cols = [c for c in df_static.columns
                       if c not in df_panel.columns or c == "gid"]
        df_panel = df_panel.merge(df_static[static_cols], on="gid", how="left")

    # Resolve country code: prefer gwno column
    gwno_col = next((c for c in df_panel.columns if c in ("gwno", "gwnoa", "gwno_a")), None)
    if gwno_col and gw_to_iso3:
        df_panel["iso3"] = pd.to_numeric(df_panel[gwno_col], errors="coerce").map(gw_to_iso3)

    # Ensure numeric types
    df_panel["year"] = pd.to_numeric(df_panel["year"], errors="coerce").astype("Int64")
    for col in ["pop_gpw_sum", "nlights_calib_mean", "rainfall_sd", "temp_sd", "ttime_mean"]:
        if col in df_panel.columns:
            df_panel[col] = pd.to_numeric(df_panel[col], errors="coerce")

    # Filter to panel years
    df_panel = df_panel[
        df_panel["year"].between(PANEL_START_YEAR, PANEL_END_YEAR)
    ].copy()

    print(f"Cell-year panel : {len(df_panel):,} rows")
    print(f"Years           : {df_panel['year'].min()}–{df_panel['year'].max()}")
    print(f"Cells           : {df_panel['gid'].nunique():,}")
    print(f"Countries (iso3): {df_panel['iso3'].nunique() if 'iso3' in df_panel.columns else 'n/a'}")
else:
    print("WARNING: yearly panel not loaded — cannot proceed with feature engineering")